# Pattern Prediction: RNN vs LSTM

Recurrent networks process sequences one step at a time, maintaining hidden
state across timesteps. This notebook compares two architectures on a binary
pattern prediction task.

- **RNN**: simple recurrence `h' = tanh(W_ih * x + W_hh * h + b)`
- **LSTM**: gated recurrence with forget/input/output gates and cell state

**CLI equivalents:** `make example-rnn` and `make example-lstm`


## Architecture

Both models use a recurrent layer followed by a linear output layer.
The `~>` operator chains layers, and `autoName` registers parameters
for gradient tracking.


In [ ]:
:t rnn

In [ ]:
:t lstm

In [ ]:
:t recurStep

## Data: Binary Pattern Prediction

`patternData` generates sequences of 1D binary values. Given a prefix,
the model must predict the next value at each timestep.

The task tests whether the recurrent model can learn temporal patterns
from 8 example sequences.


In [ ]:
-- :t patternData (lives in idris-ml-examples)


## Training: RNN

The RNN uses SGD at lr=0.03. `epochRecurrentVar` handles the
sequence unrolling: for each timestep, it feeds the input, collects the
output, computes BCE loss against the target, and backpropagates through time.


**Evaluation.** Convert the trained model to inference mode with `eval` (retypes it `WithGrad -> NoGrad`, runs tape-free), then `forward` it and read the output tensor. See `packages/idris-ml-examples/src/Example/Supervised.idr` for the idiomatic pattern.

## Training: LSTM

The LSTM uses the same task but adds a linear output layer (the LSTM
hidden state dimension doesn't need to match the output dimension).
LSTM typically converges faster on sequence tasks thanks to its gating.


**Evaluation.** Convert the trained model to inference mode with `eval` (retypes it `WithGrad -> NoGrad`, runs tape-free), then `forward` it and read the output tensor. See `packages/idris-ml-examples/src/Example/Supervised.idr` for the idiomatic pattern.

## RNN vs LSTM

Key differences:

| | RNN | LSTM |
|---|-----|------|
| Hidden state | Single vector | Cell state + hidden state |
| Gates | None | Forget, input, output |
| Long-range dependencies | Vanishing gradients | Gating preserves information |
| Parameters | Fewer | ~4x more (4 gate matrices) |

For this small task, both converge. On longer sequences (like NTM/DNC tasks),
LSTM's gating becomes essential.

idris-ml also provides `gruLayer` (Gated Recurrent Unit), which has 2 gates
instead of LSTM's 3 — a middle ground between RNN simplicity and LSTM capacity.


In [ ]:
:t gru

## PyTorch Comparison

```python
model = nn.RNN(input_size=1, hidden_size=1, batch_first=True)
# or
model = nn.LSTM(input_size=1, hidden_size=1, batch_first=True)

for epoch in range(2000):
    hidden = None
    for x_t, y_t in sequence:
        output, hidden = model(x_t, hidden)
        loss += F.binary_cross_entropy_with_logits(output, y_t)
    loss.backward()
    optimizer.step()
```

In idris-ml, `epochRecurrentVar` handles the timestep loop,
hidden state threading, and BPTT in one call.

See `pytorch/torch_ref/scripts/rnn.py` and `lstm.py` for the full references.


Next: [Transformer](transformer.ipynb) — attention-based sequence processing.
